<a href="https://colab.research.google.com/github/kritirakheja/elizabeth-bennet-llm/blob/main/01_finetuning_elizabeth_bennet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Finetuning a LLM on a Fictional Charcter

- Link to HuggingFace dataset: https://huggingface.co/datasets/KritiR/Elizabeth_question_answer
- Link to HuggingFace adapters: finetuned model

##Set Up

In [1]:
!pip install unsloth -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6

In [2]:
#HF Key

from google.colab import userdata
hf_token = userdata.get('hf_token')

##Data: from novel to Q&A pairs

This dataset was created using unsloth recipes. Read more about it here [link blog]

In [3]:
from datasets import load_dataset
from google.colab import userdata

# Load the main dataset
dataset = load_dataset("KritiR/Elizabeth_question_answer", "data", split="train", token=hf_token)

README.md:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

data/batch_00000.parquet:   0%|          | 0.00/166k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
print(dataset)

Dataset({
    features: ['chunk_text', 'source_file', 'llm_structured_1'],
    num_rows: 100
})


In [5]:
dataset['llm_structured_1'][0]

{'answer': "I make of it what I commonly make of my dear mother's matrimonial projects: that they are begun in spirits, pursued with confidence, and founded upon very slender acquaintance with the gentleman concerned. A young man with four or five thousand a year is, in her imagination, marked out at once for one of her daughters; but for my own part, I cannot suppose every new-comer enters a neighbourhood with the settled design of falling in love. I dare say Mr. Bingley may be very well, and I shall have no objection to meeting him; yet I would rather think a little less of his income, and a little more of his understanding, before I dispose of him in my thoughts.",
 'evidence_quote': '“Oh, single, my dear, to be sure! A single man of large fortune; four or five thousand a year. What a fine thing for our girls!”',
 'question': "Elizabeth, what do you make of your mother's eagerness for Mr. Bennet to visit Mr. Bingley as soon as he comes to Netherfield?"}

##Load Base Model

In [6]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


##Format Training Data

In [8]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

In [9]:
import json

def to_conversations(example):
    qa = example["llm_structured_1"]
    if isinstance(qa, str):
        qa = json.loads(qa)
    return {
        "conversations": [
            {"role": "user",      "content": qa["question"]},
            {"role": "assistant", "content": qa["answer"]},
        ]
    }

dataset = dataset.map(to_conversations)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [10]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    bos = tokenizer.bos_token
    texts = [t[len(bos):] if bos and t.startswith(bos) else t for t in texts]
    return { "text": texts }

In [11]:
# dividing the data into training and validation set

split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds   = split["test"]
print(len(train_ds), "train /", len(val_ds), "validation")

90 train / 10 validation


In [12]:
train_ds = train_ds.map(formatting_prompts_func, batched=True)
val_ds   = val_ds.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [13]:
#example
train_ds["text"][0]

"<|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nElizabeth, what did you make of your father's teasing about Mr. Bingley while you were trimming your hat, and were you at all inclined to rely on Mrs. Long for an introduction?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nMy father delights in a little archness, and I am seldom sorry to be made the object of it, when it is so harmlessly bestowed. When he spoke of Mr. Bingley liking my hat, I understood him to be diverting himself at my mother's expense more than expressing any concern for Mr. Bingley's taste. As for Mrs. Long, I thought it very possible that we might meet him at the assemblies, and that an introduction might there be obtained; but I was never of a temper to build much upon the civilities of any one person. If Mrs. Long chose to serve us, very well; if not, I dare say I could endure the disap

##Training

In [14]:
# supervised fine tuning trainer

from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset=val_ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        max_steps = -1,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        eval_strategy="steps",
        eval_steps=5
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/90 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10 [00:00<?, ? examples/s]

In [15]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [16]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nElizabeth, what induced you to mention Mr. Wickham to Mr. Darcy, and what did you make of Mr. Darcy's manner when his name was introduced?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nI own I was tempted into an imprudence. Having learnt from Mr. Wickham enough to think Mr. Darcy greatly in the wrong, I could not resist letting him know that his ill usage was not so wholly buried as he might wish. When I named our new acquaintance, I saw his countenance immediately assume a deeper air of pride and constraint; and when he spoke of Mr. Wickham's talent for making friends, without answering for his keeping them, I was confirmed in believing there had been injustice on one side, and misfortune on the other. Yet I was sensible, the next moment, that I had betrayed too much warmth, and had little 

In [17]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

"                                                                      I own I was tempted into an imprudence. Having learnt from Mr. Wickham enough to think Mr. Darcy greatly in the wrong, I could not resist letting him know that his ill usage was not so wholly buried as he might wish. When I named our new acquaintance, I saw his countenance immediately assume a deeper air of pride and constraint; and when he spoke of Mr. Wickham's talent for making friends, without answering for his keeping them, I was confirmed in believing there had been injustice on one side, and misfortune on the other. Yet I was sensible, the next moment, that I had betrayed too much warmth, and had little reason to be satisfied with my own discretion.<|eot_id|>"

In [18]:
trainer_stats = trainer.train()
print(trainer_stats)

for log in trainer.state.log_history:
    print(log)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 90 | Num Epochs = 2 | Total steps = 24
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
5,2.486979,2.378523
10,2.192741,2.057393
15,1.773569,1.970264
20,1.882422,1.929450
24,1.942798,1.917177


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=24, training_loss=2.148081804315249, metrics={'train_runtime': 193.5448, 'train_samples_per_second': 0.93, 'train_steps_per_second': 0.124, 'total_flos': 1608116759543808.0, 'train_loss': 2.148081804315249, 'epoch': 2.0})
{'loss': 2.9767770767211914, 'grad_norm': 1.7945222854614258, 'learning_rate': 0.0, 'epoch': 0.08888888888888889, 'step': 1}
{'loss': 2.907336473464966, 'grad_norm': 1.5109299421310425, 'learning_rate': 4e-05, 'epoch': 0.17777777777777778, 'step': 2}
{'loss': 2.6912648677825928, 'grad_norm': 1.4811118841171265, 'learning_rate': 8e-05, 'epoch': 0.26666666666666666, 'step': 3}
{'loss': 2.528632640838623, 'grad_norm': 1.8443019390106201, 'learning_rate': 0.00012, 'epoch': 0.35555555555555557, 'step': 4}
{'loss': 2.486978530883789, 'grad_norm': 1.0671002864837646, 'learning_rate': 0.00016, 'epoch': 0.4444444444444444, 'step': 5}
{'eval_loss': 2.378523349761963, 'eval_runtime': 2.4968, 'eval_samples_per_second': 4.005, 'eval_steps_per_second': 1.202

##Inference

In [23]:
# inference with fine tuned model

from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "What do you think of your Mr Darcy?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,          # <-- new
).to("cuda")

eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")

outputs = model.generate(
    input_ids=inputs.input_ids,        # was: input_ids = inputs
    attention_mask=inputs.attention_mask,  # <-- new
    max_new_tokens=128,
    use_cache=True,
    temperature=0.7,
    min_p=0.05,
    eos_token_id=[tokenizer.eos_token_id, eot_id],
)

print(tokenizer.decode(outputs[0], skip_special_tokens = True))

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

user

What do you think of your Mr Darcy?assistant

I have no opinion of him at all, for I do not know him well enough to form one. He appears to me a gentleman of pride and consequence, who thinks himself above his company, and who, when he does speak, says more than he means. If he is to be esteemed for his manners, his understanding, or his temper, I am not yet convinced. I have seen enough of him to be little inclined to admire him; and, for my own part, I prefer a man who can be pleasant without being proud.


In [29]:
import gc
import torch

if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer
gc.collect()
torch.cuda.empty_cache()

In [30]:
#inference with base model

from unsloth.chat_templates import get_chat_template

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
base_tokenizer = get_chat_template(base_tokenizer, chat_template = "llama-3.1")
FastLanguageModel.for_inference(base_model)

base_eot_id = base_tokenizer.convert_tokens_to_ids("<|eot_id|>")

base_inputs = base_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to("cuda")


base_outputs = base_model.generate(
    input_ids=base_inputs.input_ids,
    attention_mask=base_inputs.attention_mask,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.7,
    min_p=0.05,
    eos_token_id=[base_tokenizer.eos_token_id, base_eot_id],
)

print(base_tokenizer.decode(base_outputs[0], skip_special_tokens = True))

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_uti

system

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

user

What do you think of your Mr Darcy?assistant

You're referring to the iconic character from Jane Austen's novel "Pride and Prejudice." Mr. Darcy is a complex and intriguing character, known for his pride, wit, and ultimately, his romantic love for Elizabeth Bennet.

As a conversational AI, I don't have personal opinions or feelings, but I can analyze and discuss Mr. Darcy's character. He's often considered one of the most beloved and enduring literary characters, and his portrayal has been adapted and reinterpreted in countless films, TV shows, and stage productions.

Mr. Darcy's pride and initial dislike of Elizabeth Bennet are understandable,


##Push to Hugging Face

In [ ]:
#model.save_pretrained("elizabeth_adapter")
#tokenizer.save_pretrained("elizabeth_adapter")

In [ ]:
#model.push_to_hub("KritiR/elizabeth-bennet-adapter", token = userdata.get('hf_token'))
#tokenizer.push_to_hub("KritiR/elizabeth-bennet-adapter", token = userdata.get('hf_token'))

Read about how to evlaue a fine-tuned character model [here](https://colab.research.google.com/drive/1UBDOy6us6VZ87Xkcvdu3wsi2EoKSPzgg?usp=sharing)